In [ ]:
import os
from sentence_transformers import SentenceTransformer

# Configura tu token como variable de entorno de la sesión actual


In [7]:
import pandas as pd
import re

# 1. Cargar la base de datos (Optimizado para archivos grandes)
df = pd.read_parquet('data/dataset_50k_anonymized.parquet')

def limpiar_texto(text):
    if not isinstance(text, str):
        return ""
    try:
        text = text.encode('cp1252').decode('utf-8')
    except:
        pass 
    
    # Eliminar números (opcional, según tu solicitud)
    #text = re.sub(r'\d+', '', text)
    
    # Eliminar caracteres especiales innecesarios pero mantener signos de puntuación clave
    text = re.sub(r'[^\w\s¿?¡!.,]', '', text)
    
    # Eliminar espacios múltiples
    text = " ".join(text.split())
    
    return text

# Aplicar limpieza a la columna de entrada (input)
df['input_clean'] = df['input'].apply(limpiar_texto)

# 2. Filtrar por longitud (Eliminar mensajes de < 3 palabras)
# Esto descarta "Hola", "Ok", "Gracias"
df = df[df['input_clean'].str.split().str.len() >= 3]

print(f"Registros restantes tras la limpieza: {len(df)}")

Registros restantes tras la limpieza: 42140


In [8]:
from sentence_transformers import SentenceTransformer

# Modelo multilingüe de alta disponibilidad (No suele pedir token)
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

# Si el error persiste, añade este parámetro para forzar que ignore tokens:
# model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2', use_auth_token=False)


# Generar los embeddings
# Nota: Si tienes GPU, esto será muy rápido
embeddings = model.encode(df['input_clean'].tolist(), show_progress_bar=True)

# Ahora 'embeddings' es una matriz numérica que representa el significado de cada frase
# Puedes guardarlos para no tener que volver a procesarlos
import numpy as np
np.save('embeddings_hey_banco.npy', embeddings)

Batches: 100%|██████████| 1317/1317 [14:54<00:00,  1.47it/s]


In [13]:
from sklearn.cluster import KMeans
import pandas as pd
import numpy as np

# 1. Cargar embeddings y tu dataframe limpio
embeddings = np.load('data/embeddings_hey_banco.npy')
# Asegúrate de que df_final sea el que tiene las filas que no borraste
df_final = df

# 2. Definir cuántos grupos quieres (ej. 8 categorías de problemas)
num_clusters = 8
kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
df_final['cluster'] = kmeans.fit_predict(embeddings)

# 3. ¿Cómo saber qué significa cada cluster?
# Tomamos los 5 ejemplos más representativos de cada uno
for i in range(num_clusters):
    print(f"\n--- CLUSTER {i} (Posible Categoría) ---")
    ejemplos = df_final[df_final['cluster'] == i]['input_clean'].head(5).values
    for ej in ejemplos:
        print(f"-> {ej}")


--- CLUSTER 0 (Posible Categoría) ---
-> En donde hay un kiosko hey en CDMX para adquirir una tarjeta ?
-> Que precio tiene la tarjeta física en el kiosko hey?
-> DHL me mando una notificación que mi tarjeta está e camino pero me muestra otra dirección a la que yo proporcione. Cual es el paso que debo seguir para corregir?
-> Solo para ver eficaz si mi tarjeta ya quedó activada
-> Quiero cambiar mi tarjeta virtual, que es lo que debo hacer?

--- CLUSTER 1 (Posible Categoría) ---
-> Verificación de cuenta
-> Puedo enviar mis documentos de otra manera
-> Ok déjame checar y para solicitarlo fondo tendría q ir o comunicarme
-> Donde puedo pagar impuestos a hacienda
-> Si no hay un cajero automático de Banregio cerca de mi domicilio, hay algún otro banco que me permita realizar la misma operación?

--- CLUSTER 2 (Posible Categoría) ---
-> Hola aún se pueden hacer depósitos en Oxxo y 7 eleven ?
-> Hola no me permite hacer una compra con mi téc
-> Horario de atención?
-> Si me puedes guiar p